In [1]:
import geopandas as gpd
import pandas as pd
import os

In [2]:
# USA counties url and reading data
url = 'https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_2_counties.zip'
usa_counties = gpd.read_file(url)

# Filtering to California counties
cal_counties = (
    usa_counties.query('REGION == "CA"')[['REGION', 'NAME', 'NAME_ALT', 'geometry']]
    .rename(columns={
        "REGION": "state",
        "NAME": "county",
        "NAME_ALT": "county_name_alt"
    })
)

# Setting crs
cal_counties = cal_counties.to_crs("EPSG:4326")

# Reading housing.csv
housing_df = pd.read_csv("../data/raw/housing.csv")

# Converting housing_df to geodataframe
housing_gdf = gpd.GeoDataFrame(
    housing_df,
    geometry=gpd.points_from_xy(housing_df.longitude, housing_df.latitude),
    crs="EPSG:4326"
)

# Spatial join - housing_df and cal_counties
housing_with_county = gpd.sjoin(
    housing_gdf,
    cal_counties[['county', 'county_name_alt', 'geometry']],
    how="left",
    predicate="within"
)

housing_with_county = housing_with_county.drop(columns=['geometry', 'index_right'])

# Saving housing_with_county.csv to processed folder
housing_with_county.to_csv("../data/processed/housing_with_county.csv", index=False)

# Saving cal_counties as geojson
cal_counties.to_file("../data/raw/cal_counties.geojson", driver="GeoJSON")

In [3]:
gpd.read_file("../data/raw/cal_counties.geojson").head()

,state,county,county_name_alt,geometry
0,CA,Imperial,Imperial County,"POLYGON ((-114.72428 32.71284, -114.76454 32.7..."
1,CA,San Diego,San Diego County,"POLYGON ((-116.1054 32.60838, -116.16174 32.60..."
2,CA,Del Norte,Del Norte County,"POLYGON ((-124.06932 41.46371, -124.07608 41.4..."
3,CA,Humboldt,Humboldt County,"POLYGON ((-124.02271 40.00214, -124.03346 40.0..."
4,CA,Mendocino,Mendocino County,"POLYGON ((-123.53764 38.77233, -123.55885 38.7..."
